In [ ]:
## TASCA 1. Presentació del conjunt de dades

El conjunt de dades correspon a una clínica veterinària que recull informació sobre els serveis d'atenció a animals en diferents centres de Catalunya. 
L'objectiu de l'anàlisi és estudiar l'activitat veterinària, comparant els serveis, les espècies ateses, els centres, el nombre d'animals i la durada de les visites. 
També es vol detectar possibles errors o dades incompletes abans de fer l'anàlisi.

El fitxer conté 120 registres i les columnes següents:

- **servei**: tipus de servei veterinari realitzat(consulta, vacuna, radiografia, etc)
- **espècie**: espècie de l'animal atès(gos, gat, conill, ocells, exòtics...)
- **centre**: municipi o centre veterinari on s'ha realitzat l'atenció.
- **animals_atesos**: nombre d'animals atesos en el servei.
- **durada_min**: durada del servei en minuts.
- **data**: data en què es va realitzar l'atenció.

Les dades presenten alguns problemes de qualitat, com ara camps buits, espais sobrants, valors no numèrics i dates incorrectes. 
Per tant, abans de calcular estadístiques o crear gràfics, serà necessari netejar i validar el conjunt de dades.

In [ ]:
## Fases de l'anàlisi

1. **Exploració inicial:** càrrega del CSV, visualització de les primeres files, dimensions del conjunt de dades i tipus de dades.
2. **Detecció d'incidències:** identificació de camps obligatoris buits, valors numèrics incorrectes, dates invàlides i inconsistències de format.
3. **Neteja i validació:** aplicació de les transformacions necessàries i definició de les regles per determinar quins registres són vàlids.
4. **Gestió dels registres incorrectes:** identificació dels registres rebutjats i explicació dels motius pels quals no es poden acceptar.
5. **Anàlisi de les dades netes:** càlcul d’estadístiques i indicadors rellevants, amb agrupacions per les variables de classificació quan sigui convenient.
6. **Resultats i conclusions:** resum dels principals resultats i valoració de la qualitat de les dades.

In [ ]:
# EXERCICI 1 - Exploració inciial

import csv
from datetime import datetime

nom_fitxer = "clinica_veterinaria.csv"

with open(nom_fitxer, mode="r", encoding="utf-8-sig", newline="") as fitxer: #-sig elimina el BOM
    lector = csv.DictReader(fitxer)
    camps = lector.fieldnames
    registres = list(lector)

# Comprovacions
print("Camps del CSV:", camps)
print("Nombre total de registres:", len(registres))
if registres:
    print("Exemple de primer registre:", registres[0])



Camps del CSV: ['servei', 'especie', 'centre', 'animals_atesos', 'durada_min', 'data']
Nombre total de registres: 120
Exemple de primer registre: {'servei': 'Microxip', 'especie': 'Gos', 'centre': 'Girona', 'animals_atesos': '12', 'durada_min': '114,0', 'data': '2025-04-21'}


In [ ]:
# EXERCICI 2 - Detecció d'indidències

from datetime import datetime

buits_obligatoris = []
numerics_incorrectes = []
dates_invalides = []
inconsistencies_format = []

camps_obligatoris = ["servei", "especie", "centre"]

def es_data_valida(text: str) -> bool:
    """Comprova si una data té el format estàndard AAAA-MM-DD i és una data vàlida."""
    try:
        datetime.strptime(str(text).strip(), "%Y-%m-%d")
        return True
    except (ValueError, TypeError):
        return False

if "registres" not in globals():
    import csv

    nom_fitxer = "clinica_veterinaria.csv"
    with open(nom_fitxer, mode="r", encoding="utf-8-sig", newline="") as fitxer:
        lector = csv.DictReader(fitxer)
        registres = list(lector)

for idx, reg in enumerate(registres, start=2): # Comencem a la fila 2 del CSV
    
    # 1. Identificació de camps obligatoris buits i espais sobrants
    for camp in camps_obligatoris:
        val = reg.get(camp, "")
        if val is None or str(val).strip() == "":
            buits_obligatoris.append({"fila": idx, "camp": camp, "valor": val})
        elif val != str(val).strip():
            inconsistencies_format.append({"fila": idx, "camp": camp, "detall": f"Espais sobrants: '{val}'"})

    # 2. Identificació de valors numèrics incorrectes en 'animals_atesos'
    anim_raw = reg.get("animals_atesos", "")
    try:
        anim_val = int(str(anim_raw).strip())
        if anim_val <= 0:
            numerics_incorrectes.append({"fila": idx, "camp": "animals_atesos", "valor": anim_raw, "motiu": "Nombre <= 0"})
    except (ValueError, TypeError):
        numerics_incorrectes.append({"fila": idx, "camp": "animals_atesos", "valor": anim_raw, "motiu": "Text o caràcters invàlids"})

    # 3. Identificació de valors numèrics incorrectes i comes en 'durada_min'
    durada_raw = str(reg.get("durada_min", ""))
    if "," in durada_raw:
        inconsistencies_format.append({"fila": idx, "camp": "durada_min", "detall": f"Coma decimal utilitzada: '{durada_raw}'"})
    
    try:
        durada_val = float(durada_raw.strip().replace(",", "."))
        if durada_val <= 0:
            numerics_incorrectes.append({"fila": idx, "camp": "durada_min", "valor": durada_raw, "motiu": "Durada <= 0"})
    except (ValueError, TypeError):
        numerics_incorrectes.append({"fila": idx, "camp": "durada_min", "valor": durada_raw, "motiu": "Text o caràcters invàlids"})

    # 4. Identificació de dates invàlides
    data_raw = str(reg.get("data", "")).strip()
    if not es_data_valida(data_raw):
        dates_invalides.append({"fila": idx, "camp": "data", "valor": data_raw})

print("=== RESUM DE LA DETECCIONAL D'INCIDÈNCIES ===")
print(f"1. Camps obligatoris buits: {len(buits_obligatoris)} trobats")
print(f"2. Valors numèrics incorrectes: {len(numerics_incorrectes)} trobats")
print(f"3. Dates invàlides/incorrectes: {len(dates_invalides)} trobades")
print(f"4. Inconsistències de format (comes, espais): {len(inconsistencies_format)} trobades\n")

=== RESUM DE LA DETECCIONAL D'INCIDÈNCIES ===
1. Camps obligatoris buits: 14 trobats
2. Valors numèrics incorrectes: 24 trobats
3. Dates invàlides/incorrectes: 11 trobades
4. Inconsistències de format (comes, espais): 17 trobades

